In [ ]:
import json
import pandas as pd
from pathlib import Path

LEADERBOARD = Path('/path/to/BrainWear_Kareem/FYP/baseline/leaderboard.json')
MIN_CLASS_SUPPORT = 6

with open(LEADERBOARD) as f:
    lb = json.load(f)

df = pd.DataFrame(lb['ranked'])
df = df[df['needs_rerun'] == False]
df = df[df['f1'].notna()]

DIM_2D = {'brainwear_png', 'brats_png'}
df['dim'] = df['dataset'].apply(lambda d: '2D' if d in DIM_2D else '3D')

def get_strategy(row):
    cfg = row['config']
    if 'strategy' in cfg:
        return cfg['strategy']
    return 'regression' if cfg.get('quantile') else 'categorical'

df['strategy'] = df.apply(get_strategy, axis=1)

def _run_extras(row):
    """Return min class support and score_name for a run (single file read)."""
    try:
        p = Path(row['path'])
        if 'ae_classifier' in row['pipeline']:
            data = json.loads((p / 'cv_run_details.json').read_text())
            report = data.get('oof_results', {}).get('classification_report', {})
            score_name = data.get('arguments', {}).get('score_name')
        else:
            data = json.loads((p / 'results.json').read_text())
            report = data.get('oof_results', {}).get('classification_report', {})
            args_path = p / 'args.json'
            score_name = json.loads(args_path.read_text()).get('score_name') if args_path.exists() else None
        supports = [v['support'] for k, v in report.items()
                    if isinstance(v, dict) and k not in ('macro avg', 'weighted avg')]
        min_support = int(min(supports)) if supports else 0
    except Exception:
        min_support, score_name = 0, None
    return {'min_support': min_support, 'score_name': score_name}

extras = df.apply(_run_extras, axis=1, result_type='expand')
df['min_support'] = extras['min_support']
df['score_name'] = extras['score_name']

best = (
    df[df['min_support'] >= MIN_CLASS_SUPPORT]
    .sort_values('f1', ascending=False)
    .groupby(['dim', 'pipeline', 'dataset', 'strategy'], sort=False)
    .first()
    .reset_index()
)

cols = ['dim', 'pipeline', 'dataset', 'strategy', 'score_name', 'model', 'f1', 'balanced_accuracy', 'accuracy', 'min_support', 'run_name']
display(
    best[cols]
    .sort_values(['dim', 'strategy', 'f1'], ascending=[True, True, False])
    .style
    .format({'f1': '{:.3f}', 'balanced_accuracy': '{:.3f}', 'accuracy': '{:.3f}', 'min_support': '{:.0f}'})
    .set_caption(f'Best config per sweep  (min class support ≥ {MIN_CLASS_SUPPORT}, ranked by macro F1)')
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# df already has min_support and score_name from cell-0
best_reliable = (
    df[df['min_support'] >= MIN_CLASS_SUPPORT]
    .sort_values('f1', ascending=False)
    .groupby(['pipeline', 'dataset'], sort=False)
    .first()
    .reset_index()
    [['pipeline', 'dataset', 'score_name', 'model', 'f1', 'balanced_accuracy', 'accuracy', 'min_support', 'run_name']]
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)

display(
    best_reliable.style
    .format({'f1': '{:.3f}', 'balanced_accuracy': '{:.3f}', 'accuracy': '{:.3f}', 'min_support': '{:.0f}'})
    .bar(subset=['f1'], color='steelblue', vmin=0, vmax=1)
    .set_caption(f'Best reliable run per pipeline x dataset  (min class support ≥ {MIN_CLASS_SUPPORT})')
)

labels = [f"{r.pipeline}\n{r.dataset}" for _, r in best_reliable.iterrows()]
x = np.arange(len(labels))
width = 0.26

fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.8), 5))
ax.bar(x - width, best_reliable['f1'],                width, label='Macro F1',     color='steelblue')
ax.bar(x,          best_reliable['balanced_accuracy'], width, label='Balanced Acc', color='coral')
ax.bar(x + width,  best_reliable['accuracy'],          width, label='Accuracy',     color='mediumseagreen')
ax.axhline(1/3, color='grey', linestyle='--', linewidth=0.8, label='Random baseline (3-class)')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_title(f'Best reliable run per pipeline x dataset  (min class support ≥ {MIN_CLASS_SUPPORT})')
ax.legend()
fig.tight_layout()
plt.show()